In [0]:
# Provable Resilience Intelligence — Lakeflow SDP Pipeline
# Lakelink Fiber — Resilience Scoring Engine
# Implements: hazard x span geospatial intersection, SRLG diversity analysis, composite resilience scoring

import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

CATALOG = "cdm_tmforum"
SCHEMA = "provable_resilience_intelligence"

In [0]:
%sql
-- ============================================================
-- SILVER LAYER: Hazard x Span Spatial Intersection
-- Uses ST_Distance on WKT geometries to find spans within hazard buffer zones
-- Produces: silver_hazard_span_intersection
-- ============================================================

CREATE OR REFRESH MATERIALIZED VIEW silver_hazard_span_intersection
COMMENT 'Spans intersecting active hazard event buffer zones. Uses ST geospatial functions on WKT geometry for distance computation. Each row = one hazard threatening one span.'
AS
WITH active_hazards AS (
  SELECT hazard_id, hazard_type, severity, latitude, longitude, radius_km,
         geom_wkt, event_start, event_end, source, description
  FROM cdm_tmforum.provable_resilience_intelligence.geographic_hazard_event
  WHERE event_end IS NULL  -- only active/ongoing hazards
),
span_midpoints AS (
  -- Extract midpoint of each span LINESTRING for distance calculation
  SELECT span_id, conduit_id, length_km, geom_wkt,
         -- Parse midpoint from WKT LINESTRING
         ST_X(ST_Centroid(ST_GeomFromWKT(geom_wkt))) as span_mid_lon,
         ST_Y(ST_Centroid(ST_GeomFromWKT(geom_wkt))) as span_mid_lat
  FROM cdm_tmforum.provable_resilience_intelligence.physical_resource_span
  WHERE is_active = true AND geom_wkt IS NOT NULL
)
SELECT
  h.hazard_id,
  h.hazard_type,
  h.severity as hazard_severity,
  s.span_id,
  s.conduit_id,
  -- Haversine great-circle distance in km (BUG-QA-005 fix: ST_Distance returns degrees, not km)
  6371 * 2 * ASIN(SQRT(
    POWER(SIN(RADIANS(s.span_mid_lat - h.latitude) / 2), 2) +
    COS(RADIANS(h.latitude)) * COS(RADIANS(s.span_mid_lat)) *
    POWER(SIN(RADIANS(s.span_mid_lon - h.longitude) / 2), 2)
  )) as distance_km,
  h.radius_km as hazard_radius_km,
  CASE
    WHEN 6371 * 2 * ASIN(SQRT(
      POWER(SIN(RADIANS(s.span_mid_lat - h.latitude) / 2), 2) +
      COS(RADIANS(h.latitude)) * COS(RADIANS(s.span_mid_lat)) *
      POWER(SIN(RADIANS(s.span_mid_lon - h.longitude) / 2), 2)
    )) <= h.radius_km * 0.5 THEN 'direct_impact'
    WHEN 6371 * 2 * ASIN(SQRT(
      POWER(SIN(RADIANS(s.span_mid_lat - h.latitude) / 2), 2) +
      COS(RADIANS(h.latitude)) * COS(RADIANS(s.span_mid_lat)) *
      POWER(SIN(RADIANS(s.span_mid_lon - h.longitude) / 2), 2)
    )) <= h.radius_km THEN 'buffer_zone'
    ELSE 'proximity_alert'
  END as impact_zone,
  h.event_start,
  h.source as hazard_source,
  h.description as hazard_description,
  current_timestamp() as computed_at
FROM active_hazards h
JOIN span_midpoints s
  -- Haversine bounding box pre-filter: degrees ≈ km / 111 latitude, / (111 * cos(lat)) longitude
  ON ABS(s.span_mid_lat - h.latitude) < h.radius_km * 1.5 / 111.0
  AND ABS(s.span_mid_lon - h.longitude) < h.radius_km * 1.5 / (111.0 * COS(RADIANS(h.latitude)))
WHERE 6371 * 2 * ASIN(SQRT(
    POWER(SIN(RADIANS(s.span_mid_lat - h.latitude) / 2), 2) +
    COS(RADIANS(h.latitude)) * COS(RADIANS(s.span_mid_lat)) *
    POWER(SIN(RADIANS(s.span_mid_lon - h.longitude) / 2), 2)
  )) <= h.radius_km * 1.5  -- haversine distance in km; include 50% buffer beyond declared radius

In [0]:
%sql
-- ============================================================
-- SILVER LAYER: SRLG Diversity Analysis
-- For each diverse service, compare working vs protect path SRLGs
-- Detects "false diversity" — services sold as diverse but sharing SRLGs
-- ============================================================

CREATE OR REFRESH MATERIALIZED VIEW silver_srlg_diversity_analysis
COMMENT 'SRLG diversity analysis per service. Identifies services sold as diverse whose working and protect paths share common SRLGs (false diversity). Critical for Sales verification and Service Assurance.'
AS
WITH working_srlgs AS (
  -- SRLGs on working paths (BUG-QA-027 fix: uses unified view with 931 SRLGs = 809 seed + 122 derived)
  SELECT sp.service_id, sg.srlg_id, sg.srlg_type, sg.severity
  FROM cdm_tmforum.provable_resilience_intelligence.service_path sp
  JOIN cdm_tmforum.provable_resilience_intelligence.unified_shared_risk_group sg ON sp.span_id = sg.span_id
  WHERE sp.path_type = 'working'
),
protect_srlgs AS (
  -- SRLGs on protect paths (BUG-QA-027 fix: uses unified view with 931 SRLGs = 809 seed + 122 derived)
  SELECT sp.service_id, sg.srlg_id, sg.srlg_type, sg.severity
  FROM cdm_tmforum.provable_resilience_intelligence.service_path sp
  JOIN cdm_tmforum.provable_resilience_intelligence.unified_shared_risk_group sg ON sp.span_id = sg.span_id
  WHERE sp.path_type = 'protect'
),
shared_srlgs AS (
  -- SRLGs that appear on BOTH working and protect paths = false diversity
  SELECT w.service_id, w.srlg_id, w.srlg_type, w.severity,
         'shared' as diversity_status
  FROM working_srlgs w
  INNER JOIN protect_srlgs p ON w.service_id = p.service_id AND w.srlg_id = p.srlg_id
),
service_diversity AS (
  SELECT
    si.service_id,
    si.customer_id,
    si.service_type,
    si.bandwidth,
    si.sold_as_diverse,
    si.mrc_usd,
    COUNT(DISTINCT ws.srlg_id) as working_srlg_count,
    COUNT(DISTINCT ps.srlg_id) as protect_srlg_count,
    COUNT(DISTINCT ss.srlg_id) as shared_srlg_count,
    COLLECT_SET(ss.srlg_type) as shared_srlg_types,
    MAX(CASE ss.severity WHEN 'critical' THEN 4 WHEN 'high' THEN 3 WHEN 'medium' THEN 2 ELSE 1 END) as max_shared_severity
  FROM cdm_tmforum.provable_resilience_intelligence.service_inventory si
  LEFT JOIN working_srlgs ws ON si.service_id = ws.service_id
  LEFT JOIN protect_srlgs ps ON si.service_id = ps.service_id
  LEFT JOIN shared_srlgs ss ON si.service_id = ss.service_id
  WHERE si.sold_as_diverse = true
  GROUP BY si.service_id, si.customer_id, si.service_type, si.bandwidth, si.sold_as_diverse, si.mrc_usd
)
SELECT *,
  CASE
    WHEN shared_srlg_count = 0 THEN 'truly_diverse'
    WHEN shared_srlg_count <= 2 AND max_shared_severity <= 2 THEN 'minor_overlap'
    WHEN shared_srlg_count <= 5 THEN 'significant_overlap'
    ELSE 'false_diversity'
  END as diversity_verdict,
  -- Diversity score: 100 = fully diverse, 0 = completely shared
  CASE
    WHEN working_srlg_count + protect_srlg_count = 0 THEN 100.0
    ELSE round(100.0 * (1.0 - (2.0 * shared_srlg_count) / (working_srlg_count + protect_srlg_count + 0.001)), 1)
  END as path_diversity_score,
  current_timestamp() as computed_at
FROM service_diversity

In [0]:
%sql
-- ============================================================
-- GOLD LAYER: Composite Resilience Score Snapshot (V2)
-- P28-DE-SCORING-PIPELINE-CONDUIT (2026-10-24): aligned to live resilience_score_snapshot v22 logic
--   * Hazards resolved at CONDUIT level via hazard_span_intersection.affected_service_ids (BUG-QA-060)
--   * Active hazard = is_within_radius AND hazard_start <= score_time AND (hazard_end NULL OR > score_time)
--   * V2 formula (DEC-PRI-HAZARD-HL20-V3): hazard = 100*exp(-n*ln2/20); telemetry = 100*(1-PERCENT_RANK(alarms));
--     overall = LEAST(100, 0.4*path + 0.3*hazard + 0.3*telemetry + SLA boost [+5 premium_plus / +2 premium])
--   * revenue_at_risk_usd = mrc_usd * (1 - overall_score/100) from the V2 score
--   * Scores ALL services (8,000); non-diverse services default to path_diversity 100 / srlg_count 0
-- P32-DE-PIPELINE-PIN-PATH-COMPONENT (2026-10-25, DEC-PRI-PATH-DIVERSITY-SOURCE):
--   * path_diversity_score and srlg_count are read from the resilience_score_snapshot path component
--     (data = v22; v23/v24 are comment-only CHANGE COLUMN commits). This is the documented live value.
--   * silver_srlg_diversity_analysis is NOT used for scoring (only 49/2,395 diverse services reproduce).
--     It stays in the pipeline as an analysis view. Re-derivation from current SRLGs is deferred to
--     P32-DE-PATH-DIVERSITY-REDERIVE-POSTDEMO, which needs a new write-window decision.
-- Dry-run validation: reproduces snapshot v22 8,000/8,000 on overall, hazards, telemetry and revenue at risk.
-- Do NOT run during the PRI demo freeze without a @pm-approved write window.
-- ============================================================

CREATE OR REFRESH MATERIALIZED VIEW gold_resilience_score
COMMENT 'Lakelink Fiber gold-layer composite resilience score per service (V2). 0.4 path diversity + 0.3 hazard exposure (conduit-level active hazards, half-life 20) + 0.3 telemetry health (alarm percentile) + SLA boost, capped at 100. revenue_at_risk_usd = mrc_usd * (1 - overall_score/100).'
AS
WITH services AS (
  SELECT si.service_id, si.mrc_usd, COALESCE(si.sla_tier, 'standard') AS sla_tier
  FROM cdm_tmforum.provable_resilience_intelligence.service_inventory si
),
diversity AS (
  -- DEC-PRI-PATH-DIVERSITY-SOURCE: pinned path component from the governed snapshot (v22 data).
  -- The snapshot holds one row per service (8,000 rows, a single score_time). The QUALIFY guards against future multi-snapshot loads.
  SELECT service_id, path_diversity_score, srlg_count AS shared_srlg_count
  FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot
  QUALIFY ROW_NUMBER() OVER (PARTITION BY service_id ORDER BY score_time DESC) = 1
),
score_clock AS (SELECT current_timestamp() AS score_time),
hazard_exposure AS (
  -- Conduit-level resolution: span -> conduit -> every service_path on that conduit (governed HSI table)
  SELECT svc AS service_id, COUNT(DISTINCT h.hazard_id) AS active_hazard_count
  FROM cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection h
  CROSS JOIN score_clock c
  LATERAL VIEW explode(split(h.affected_service_ids, ',')) e AS svc
  WHERE h.is_within_radius
    AND h.hazard_start <= c.score_time
    AND (h.hazard_end IS NULL OR h.hazard_end > c.score_time)
    AND h.affected_service_ids IS NOT NULL AND svc <> ''
  GROUP BY svc
),
alarm_health AS (
  SELECT sp.service_id, COUNT(DISTINCT a.alarm_id) AS active_alarm_count
  FROM cdm_tmforum.provable_resilience_intelligence.service_path sp
  JOIN cdm_tmforum.provable_resilience_intelligence.resource_alarm a ON sp.span_id = a.span_id
  WHERE a.cleared_at IS NULL
  GROUP BY sp.service_id
),
components AS (
  SELECT
    s.service_id, s.mrc_usd, c.score_time,
    COALESCE(d.path_diversity_score, 100.0) AS path_diversity_score,
    COALESCE(d.shared_srlg_count, 0) AS srlg_count,
    COALESCE(he.active_hazard_count, 0) AS active_hazards,
    COALESCE(ah.active_alarm_count, 0) AS active_alarms,
    ROUND(100.0 * EXP(-COALESCE(he.active_hazard_count, 0) * LN(2) / 20.0), 2) AS hazard_exposure_score,
    ROUND(100.0 * (1.0 - PERCENT_RANK() OVER (ORDER BY COALESCE(ah.active_alarm_count, 0) ASC)), 2) AS telemetry_health_score,
    CASE s.sla_tier WHEN 'premium_plus' THEN 5.0 WHEN 'premium' THEN 2.0 ELSE 0.0 END AS sla_boost
  FROM services s
  CROSS JOIN score_clock c
  LEFT JOIN diversity d ON s.service_id = d.service_id
  LEFT JOIN hazard_exposure he ON s.service_id = he.service_id
  LEFT JOIN alarm_health ah ON s.service_id = ah.service_id
),
scored AS (
  SELECT *,
    -- V2 composite defined ONCE (BUG-QA-010)
    LEAST(100.0, ROUND(ROUND(0.4 * path_diversity_score + 0.3 * hazard_exposure_score + 0.3 * telemetry_health_score, 2) + sla_boost, 2)) AS overall_score
  FROM components
)
SELECT
  concat('SNAP-', date_format(score_time, 'yyyyMMdd-HHmm'), '-', service_id) AS snapshot_id,
  service_id, score_time,
  overall_score, path_diversity_score, hazard_exposure_score, telemetry_health_score,
  srlg_count, active_hazards, active_alarms,
  ROUND(mrc_usd * (1 - overall_score / 100.0), 2) AS revenue_at_risk_usd,
  current_timestamp() AS created_at,
  'DERIVED' AS origin
FROM scored

In [0]:
%sql
-- ============================================================
-- GOLD LAYER: Revenue at Risk View
-- Joins resilience scores with customer MRC and SLA tier
-- P32-DE-PIPELINE-PIN-PATH-COMPONENT: critical threshold aligned to the KPI definition (overall_score < 40)
-- ============================================================

CREATE OR REFRESH MATERIALIZED VIEW gold_revenue_at_risk
COMMENT 'Revenue at risk per customer, aggregated from service-level resilience scores. Prioritizes Platinum customers with low resilience and high MRC exposure.'
AS
SELECT
  c.customer_id,
  c.customer_name,
  c.tier as customer_tier,
  c.industry,
  c.sla_tier,
  c.mrc_usd as customer_mrc,
  COUNT(DISTINCT rs.service_id) as service_count,
  round(AVG(rs.overall_score), 1) as avg_resilience_score,
  round(MIN(rs.overall_score), 1) as min_resilience_score,
  SUM(rs.revenue_at_risk_usd) as total_revenue_at_risk,
  SUM(rs.active_hazards) as total_active_hazards,
  SUM(rs.srlg_count) as total_shared_srlgs,
  SUM(CASE WHEN rs.overall_score < 40 THEN 1 ELSE 0 END) as critical_services,
  SUM(CASE WHEN rs.overall_score < 70 THEN 1 ELSE 0 END) as at_risk_services,
  current_timestamp() as computed_at
FROM LIVE.gold_resilience_score rs
JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory si ON rs.service_id = si.service_id
JOIN cdm_tmforum.provable_resilience_intelligence.customer c ON si.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name, c.tier, c.industry, c.sla_tier, c.mrc_usd

In [0]:
%sql
-- ============================================================
-- GOLD LAYER: Proactive Action Queue
-- Triggers reroutes, dispatches, notifications, and splice pre-positioning
-- based on resilience score thresholds and hazard/alarm conditions
-- ============================================================

CREATE OR REFRESH MATERIALIZED VIEW gold_proactive_action
COMMENT 'Proactive action queue — reroutes, crew dispatches, customer notifications, and NOC escalations triggered by resilience score thresholds. Turns resilience intelligence into operational response for Lakelink Fiber.'
AS
WITH scored_services AS (
  SELECT
    rs.service_id, rs.overall_score, rs.path_diversity_score,
    rs.hazard_exposure_score, rs.telemetry_health_score,
    rs.srlg_count, rs.active_hazards, rs.active_alarms, rs.revenue_at_risk_usd,
    si.sla_tier, si.customer_id, si.service_type
  FROM LIVE.gold_resilience_score rs
  JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory si
    ON rs.service_id = si.service_id
)
SELECT
  CONCAT('ACT-', action_type_code, '-', service_id) as action_id,
  service_id,
  action_type,
  trigger_reason,
  priority,
  'pending' as status,
  assigned_to,
  current_timestamp() as created_at,
  CAST(NULL AS TIMESTAMP) as resolved_at,
  CAST(NULL AS STRING) as resolution_notes
FROM (
  -- Critical: score < 40 → emergency reroute
  SELECT service_id, 'reroute' as action_type, 'REROUTE' as action_type_code,
    CONCAT('Resilience score ', ROUND(overall_score, 1), ' < 40. ',
           active_hazards, ' active hazards, ', active_alarms, ' alarms. ',
           'Revenue at risk: $', ROUND(revenue_at_risk_usd, 0)) as trigger_reason,
    'critical' as priority, 'NOC-Tier3' as assigned_to
  FROM scored_services WHERE overall_score < 40

  UNION ALL

  -- Critical: premium_plus with score < 40 → NOC escalation
  SELECT service_id, 'escalate_noc', 'ESC',
    CONCAT('Premium Plus SLA. Score ', ROUND(overall_score, 1), '. ',
           srlg_count, ' shared SRLGs. $', ROUND(revenue_at_risk_usd, 0), ' at risk.'),
    'critical', 'NOC-Manager'
  FROM scored_services WHERE overall_score < 40 AND sla_tier = 'premium_plus'

  UNION ALL

  -- High: score 40-60 with hazards → dispatch crew
  SELECT service_id, 'dispatch_crew', 'DISPATCH',
    CONCAT('High risk (score ', ROUND(overall_score, 1), '). ',
           active_hazards, ' active hazards near path. Pre-position splice crew.'),
    'high', 'Field-Ops-Dispatch'
  FROM scored_services WHERE overall_score BETWEEN 40 AND 59 AND active_hazards > 5

  UNION ALL

  -- High: premium_plus customers → proactive notification
  SELECT service_id, 'notify_customer', 'NTFY',
    CONCAT('Proactive alert: resilience score ', ROUND(overall_score, 1), '. ',
           active_alarms, ' active alarms. Mitigation in progress.'),
    'high', 'Customer-Success'
  FROM scored_services WHERE overall_score BETWEEN 40 AND 59 AND sla_tier = 'premium_plus'

  UNION ALL

  -- Medium: SRLG overlap detected → pre-position splice team
  SELECT service_id, 'preposition_splice', 'SPLICE',
    CONCAT('Diversity gap: ', srlg_count, ' shared SRLGs. ',
           'Path diversity score: ', ROUND(path_diversity_score, 1), '.'),
    'medium', 'Plant-Engineering'
  FROM scored_services WHERE srlg_count > 0 AND overall_score < 60
)